### 1. Konfigurasi Awal
Tahap ini dilakukan untuk mengimpor library yang dibutuhkan dan mengatur path file yang benar.

In [ ]:
import os
import time
import json
import base64
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv
from ua_parser import OS

load_dotenv()

client = OpenAI(api_key=os.getenv("GPT_KEY"))

BASE_DIR = os.getenv("BASE_DIR")
TRAIN_CSV = os.path.join(BASE_DIR, "Balanced_TrainLabels.csv")
TRAIN_DIR = os.path.join(BASE_DIR, "Train")

OUTPUT_JSONL = os.getenv("OUTPUT")

### 2. Encoding & Rekayasa Prompt
Menyusun prompt dengan skala dari 0-3 untuk setiap kelas pada masing-masing label.

In [9]:
def encode_image(image_path):
    """Mengonversi gambar ke format Base64 untuk dikirim via API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def create_gpt_prompt(boredom, engagement, confusion, frustration):
    """Membuat prompt analisis berdasarkan nilai ground-truth."""
    return f"""
Kamu adalah seorang ahli analisis mikro-ekspresi wajah dan bahasa tubuh.
Saya akan memberikan gambar seorang siswa yang sedang menatap layar komputer.

Nilai ground-truth (skala 0-3) untuk ekspresi siswa ini sudah ditetapkan:
- Boredom: {boredom}
- Engagement: {engagement}
- Confusion: {confusion}
- Frustration: {frustration}

Tugasmu:
Buatkan 1 paragraf singkat (maksimal 3 kalimat) yang MENDESKRIPSIKAN MENGAPA siswa tersebut mendapatkan nilai-nilai di atas. 
Fokus HANYA pada ciri fisik yang terlihat di gambar (misal: arah mata, posisi alis, bentuk mulut, postur tubuh).
JANGAN menganalisis hal yang tidak ada di gambar. JANGAN sebutkan angkanya di dalam paragraf deskripsimu.

Di akhir paragraf, berikan kesimpulan baku dengan format persis seperti ini:
"Berdasarkan pengamatan visual tersebut, tingkat ekspresinya adalah: Boredom: {boredom}, Engagement: {engagement}, Confusion: {confusion}, Frustration: {frustration}."
"""

### 3. Proses Generate Dataset (Format Qwen Unsloth)
Fungsi ini akan melakukan iterasi pada CSV dan memanggil API GPT-4o-mini untuk menghasilkan *caption*. 


In [10]:
def generate_dataset():
    print("[INFO] Membaca file CSV...")
    df = pd.read_csv(TRAIN_CSV)
    
    processed_images = set()
    if os.path.exists(OUTPUT_JSONL):
        with open(OUTPUT_JSONL, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                img_path = data['messages'][0]['content'][0]['image']
                processed_images.add(os.path.basename(img_path))
                
    print(f"[INFO] Melewati {len(processed_images)} gambar yang sudah diproses.")

    with open(OUTPUT_JSONL, 'a', encoding='utf-8') as outfile:
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Generate Captions GPT"):
            img_name = str(row['Image_Name'])
            if not (img_name.endswith('.jpg') or img_name.endswith('.png')): 
                img_name += '.jpg'
            
            if img_name in processed_images:
                continue
                
            img_path = os.path.join(TRAIN_DIR, img_name)
            if not os.path.exists(img_path):
                continue
                
            b, e, c, f_val = row['Boredom'], row['Engagement'], row['Confusion'], row['Frustration']
            
            success = False
            while not success:
                try:
                    
                    base64_image = encode_image(img_path)
                    gpt_prompt = create_gpt_prompt(b, e, c, f_val)
                    
                    response = client.chat.completions.create(
                        model="gpt-4o-mini",
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {"type": "text", "text": gpt_prompt},
                                    {
                                        "type": "image_url",
                                        "image_url": {
                                            "url": f"data:image/jpeg;base64,{base64_image}"
                                        }
                                    }
                                ]
                            }
                        ],
                        max_tokens=250 
                    )
                    
                    caption = response.choices[0].message.content.strip()
                    
                    user_instruction = "Analisis ekspresi wajah dan postur tubuh siswa ini secara detail. Tentukan tingkat kebosanan (boredom), ketertarikan (engagement), kebingungan (confusion), dan frustrasinya (frustration)."
                    
                    qwen_data = {
                        "messages": [
                            {
                                "role": "user",
                                "content": [
                                    {"type": "image", "image": img_path},
                                    {"type": "text", "text": user_instruction}
                                ]
                            },
                            {
                                "role": "assistant",
                                "content": caption 
                            }
                        ]
                    }
                    
                    outfile.write(json.dumps(qwen_data) + '\n')
                    outfile.flush()
                    
                    success = True 
                    time.sleep(0.5) 
                    
                except Exception as err:
                    print(f"\n[ERROR] Gagal memproses {img_name}: {err}")
                    print("Mencoba ulang dalam 5 detik...")
                    time.sleep(5)

if __name__ == "__main__":
    generate_dataset()

[INFO] Membaca file CSV...
[INFO] Melewati 0 gambar yang sudah diproses.


Generate Captions GPT: 100%|██████████| 1982/1982 [2:22:13<00:00,  4.31s/it]  
